## Import data into Databricks:

### Load raw badminton players data from CSV into a Spark DataFrame:

In [0]:
csv_path = "/Volumes/workspace/default/datasets/1. Dataset.csv"

badminton_df = spark.read.csv(csv_path, header=True, inferSchema=True)
display(badminton_df)

In [0]:
badminton_df.printSchema()

### Load the data into a Delta table using a Python notebook, saving it to your schema: General.<login>.<table>:

In [0]:
dbutils.widgets.text("login", "mariia_test", "Your login (schema name)")
login = dbutils.widgets.get("login")

catalog = "workspace"
schema = login

print(f"Working in: {catalog}.{schema}")

Create the target schema if it doesn't exist yet:

In [0]:
catalog = "workspace"
#catalog = "General"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
(
    badminton_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.badminton_players")
)

print(f"Saved to {catalog}.{schema}.badminton_players")

In [0]:
badminton = spark.table(f"{catalog}.{schema}.badminton_players")
badminton.printSchema()

### Practice basic Spark DataFrame operations (select, filter, join, groupBy)

Select only the columns needed for basic player overview:

In [0]:
basic = badminton.select("Name", "Nationality", "Age", "Rank", "Points")
display(basic)

Filter for young top-ranked players (age < 25, rank in top 20):

In [0]:
young_top_players = badminton.filter((badminton.Age < 25) & (badminton.Rank <= 20))
display(young_top_players)

Aggregate player stats by nationality: count, avg age, avg points:

In [0]:
from pyspark.sql import functions as F

stats_by_country = (
    badminton.groupBy("Nationality")
    .agg(
        F.count("*").alias("players_count"),
        F.round(F.avg("Age"), 1).alias("avg_age"),
        F.round(F.avg("Points"), 1).alias("avg_points")
    )
    .orderBy(F.desc("players_count"))
)
display(stats_by_country)

Join with a manual continent lookup table to enrich nationality data:

In [0]:
continent_lookup = spark.createDataFrame(
    [
        ("Denmark", "Europe"), ("China", "Asia"), ("Indonesia", "Asia"),
        ("Japan", "Asia"), ("Thailand", "Asia"), ("India", "Asia"),
        ("Singapore", "Asia"), ("Malaysia", "Asia"), ("Hong Kong", "Asia"),
        ("Taiwan", "Asia"), ("France", "Europe"), ("Canada", "North America"),
        ("South Korea", "Asia"), ("Netherlands", "Europe"), ("Belgium", "Europe"),
        ("Ireland", "Europe"), ("Brazil", "South America"), ("Guatemala", "North America"),
    ],
    ["Nationality", "Continent"]
)

badminton_with_continent = badminton.join(continent_lookup, on="Nationality", how="left")
display(badminton_with_continent.select("Name", "Nationality", "Continent", "Age", "Points"))